# Zain Telecom Customer 360 SQL Agent - Google Colab Notebook

This notebook connects your synthetic Zain Telecom Customer 360 SQLite database to a LangChain `create_agent` SQL agent.

It uses:

- Google Colab Secrets for `OPENAI_API_KEY`
- SQLite database file: `zain_customer_360_ai_demo.db`
- LangChain `create_agent`
- LangGraph `InMemorySaver` for short-term chat memory
- Optional Gradio UI for interactive demo

Before running:
1. Add your OpenAI key in Colab Secrets with the name `OPENAI_API_KEY`.
2. Upload `zain_customer_360_ai_demo.db` when the notebook asks for it, or place it at `/content/data/zain_customer_360_ai_demo.db`.

In [ ]:
# ============================================================
# 1. Install dependencies
# ============================================================

!pip install -U langchain langchain-openai langgraph openai gradio pandas

In [ ]:
# ============================================================
# 2. Load OpenAI API key from Google Colab Secrets
# ============================================================

import os
from pathlib import Path

try:
    from google.colab import userdata
    openai_key = userdata.get("OPENAI_API_KEY")
except Exception:
    openai_key = None

if not openai_key:
    raise ValueError(
        "OPENAI_API_KEY not found. In Colab, open the left sidebar -> Secrets -> "
        "add a secret named OPENAI_API_KEY."
    )

os.environ["OPENAI_API_KEY"] = openai_key

# You can change this if needed.
# Example:
# MODEL_NAME = "openai:gpt-5.5"
MODEL_NAME = os.getenv("MODEL_NAME", "openai:gpt-5.4")

print("OpenAI API key loaded from Colab Secrets.")
print("Model:", MODEL_NAME)

In [ ]:
# ============================================================
# 3. Locate or upload the SQLite database
# ============================================================

import shutil
import zipfile
from google.colab import files

TARGET_DB_PATH = Path("/content/data/zain_customer_360_ai_demo.db")
TARGET_DB_PATH.parent.mkdir(parents=True, exist_ok=True)

def is_sqlite_database_file(path):
    path = Path(path)
    if not path.exists() or not path.is_file():
        return False

    try:
        with open(path, "rb") as f:
            return f.read(16) == b"SQLite format 3\x00"
    except Exception:
        return False


def find_existing_db():
    candidates = [
        TARGET_DB_PATH,
        Path("/content/zain_customer_360_ai_demo.db"),
        Path("/content/zain_telecom_customer_360_database/data/zain_customer_360_ai_demo.db"),
    ]

    for candidate in candidates:
        if is_sqlite_database_file(candidate):
            return candidate

    # Search broadly inside /content
    for candidate in Path("/content").rglob("zain_customer_360_ai_demo.db"):
        if is_sqlite_database_file(candidate):
            return candidate

    for candidate in Path("/content").rglob("*.db"):
        if is_sqlite_database_file(candidate):
            return candidate

    for candidate in Path("/content").rglob("*.sqlite"):
        if is_sqlite_database_file(candidate):
            return candidate

    return None


def upload_and_prepare_db():
    print("Database not found at /content/data/zain_customer_360_ai_demo.db")
    print("Please upload either:")
    print("- zain_customer_360_ai_demo.db")
    print("- zain_customer_360_ai_demo.sqlite")
    print("- the full ZIP package that contains the DB")

    uploaded = files.upload()

    for filename in uploaded.keys():
        uploaded_path = Path("/content") / filename

        if filename.lower().endswith(".zip"):
            extract_dir = Path("/content/uploaded_zain_package")
            extract_dir.mkdir(parents=True, exist_ok=True)

            with zipfile.ZipFile(uploaded_path, "r") as zip_ref:
                zip_ref.extractall(extract_dir)

            found = find_existing_db()
            if found:
                shutil.copyfile(found, TARGET_DB_PATH)
                return TARGET_DB_PATH

        elif filename.lower().endswith((".db", ".sqlite", ".sqlite3")):
            if is_sqlite_database_file(uploaded_path):
                shutil.copyfile(uploaded_path, TARGET_DB_PATH)
                return TARGET_DB_PATH
            else:
                raise ValueError(
                    f"The uploaded file {filename} is not a valid SQLite database. "
                    "A real SQLite DB must start with the header: SQLite format 3"
                )

    found = find_existing_db()
    if found:
        shutil.copyfile(found, TARGET_DB_PATH)
        return TARGET_DB_PATH

    raise FileNotFoundError(
        "Could not find a valid SQLite database after upload. "
        "Please upload zain_customer_360_ai_demo.db."
    )


existing = find_existing_db()

if existing:
    if existing != TARGET_DB_PATH:
        shutil.copyfile(existing, TARGET_DB_PATH)
    DB_PATH = TARGET_DB_PATH
else:
    DB_PATH = upload_and_prepare_db()

print("Database ready:", DB_PATH)
print("Database size:", f"{DB_PATH.stat().st_size:,}", "bytes")

with open(DB_PATH, "rb") as f:
    print("SQLite header:", f.read(16))

In [ ]:
# ============================================================
# 4. Inspect database tables
# ============================================================

import sqlite3
import pandas as pd

conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

cursor.execute("""
SELECT name
FROM sqlite_master
WHERE type='table'
  AND name NOT LIKE 'sqlite_%'
ORDER BY name;
""")

tables = [row[0] for row in cursor.fetchall()]

table_counts = []

for table in tables:
    cursor.execute(f"SELECT COUNT(*) FROM {table}")
    table_counts.append({"table": table, "rows": cursor.fetchone()[0]})

conn.close()

pd.DataFrame(table_counts)

In [ ]:
# ============================================================
# 5. Build database schema text for the agent
# ============================================================

def get_database_schema(db_path):
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    cursor.execute("""
    SELECT name
    FROM sqlite_master
    WHERE type='table'
      AND name NOT LIKE 'sqlite_%'
    ORDER BY name;
    """)

    tables = [row[0] for row in cursor.fetchall()]
    lines = []

    for table in tables:
        lines.append(f"\nTable: {table}")
        cursor.execute(f"PRAGMA table_info({table});")

        for col in cursor.fetchall():
            _, name, col_type, notnull, _, pk = col

            flags = []
            if pk:
                flags.append("PRIMARY KEY")
            if notnull:
                flags.append("NOT NULL")

            flag_text = f" ({', '.join(flags)})" if flags else ""
            lines.append(f"- {name}: {col_type}{flag_text}")

    conn.close()
    return "\n".join(lines)


DATABASE_SCHEMA = get_database_schema(DB_PATH)

print(DATABASE_SCHEMA[:4000])
print("\n... schema loaded ...")

In [ ]:
# ============================================================
# 6. Create safe read-only SQL execution tool
# ============================================================

import re
from langchain.tools import tool

def strip_sql_code_fences(query: str) -> str:
    query = query.strip()

    if query.startswith("```"):
        query = re.sub(r"^```(?:sql)?", "", query, flags=re.IGNORECASE).strip()
        query = re.sub(r"```$", "", query).strip()

    return query


def is_read_only_sql(query: str) -> bool:
    cleaned = strip_sql_code_fences(query)
    cleaned = re.sub(r"/\*.*?\*/", "", cleaned, flags=re.DOTALL)
    cleaned = re.sub(r"--.*?$", "", cleaned, flags=re.MULTILINE)
    cleaned = cleaned.strip().lower()

    allowed_starts = ("select", "with", "pragma", "explain")

    if not cleaned.startswith(allowed_starts):
        return False

    blocked_keywords = [
        "insert ",
        "update ",
        "delete ",
        "drop ",
        "alter ",
        "create ",
        "replace ",
        "truncate ",
        "attach ",
        "detach ",
        "vacuum",
        "reindex",
    ]

    return not any(keyword in cleaned for keyword in blocked_keywords)


def rows_to_markdown(columns, rows, max_rows: int = 50) -> str:
    if not rows:
        return "Query executed successfully, but returned no rows."

    rows = rows[:max_rows]

    def clean_cell(value):
        if value is None:
            return ""
        return str(value).replace("\n", " ").replace("|", "\\|")

    header = "| " + " | ".join(columns) + " |"
    separator = "| " + " | ".join(["---"] * len(columns)) + " |"
    body = ["| " + " | ".join(clean_cell(v) for v in row) + " |" for row in rows]

    return "\n".join([header, separator] + body)


@tool
def execute_sql(query: str) -> str:
    """
    Execute a read-only SQLite query against the synthetic Zain Telecom Customer 360 database.
    Only SELECT, WITH, PRAGMA, and EXPLAIN queries are allowed.
    """

    query = strip_sql_code_fences(query)

    if not is_read_only_sql(query):
        return "Blocked for safety. Only read-only SQL is allowed."

    try:
        conn = sqlite3.connect(DB_PATH)
        cursor = conn.cursor()
        cursor.execute(query)

        rows = cursor.fetchall()
        columns = [desc[0] for desc in cursor.description] if cursor.description else []

        conn.close()

        if not columns:
            return "Query executed successfully."

        result = rows_to_markdown(columns, rows)

        if len(rows) > 50:
            result += f"\n\nShowing first 50 rows out of {len(rows)} rows."

        return result

    except Exception as e:
        return f"SQL execution error: {str(e)}"

In [ ]:
# ============================================================
# 7. Create LangChain SQL agent with memory
# ============================================================

from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

SYSTEM_PROMPT = f"""
You are a telecom data analyst working with a synthetic Zain Telecom Jordan customer 360 SQLite database.

Your job:
- Understand business questions about telecom customers.
- Write correct SQLite queries.
- Use the execute_sql tool to query the database.
- Explain results clearly for business users.
- Use conversation memory for follow-up questions.

Important rules:
- Use only read-only SQL.
- Never modify the database.
- Prefer clear joins and explain the SQL logic briefly.
- If the database does not contain enough information, say that clearly.
- This is synthetic demo data, not real customer data.

The database covers:
- Customers
- Accounts
- Subscriptions
- Plans
- Devices
- SIM cards
- Call detail records
- Data usage sessions
- SMS and roaming usage
- Top-ups
- Invoices
- Payments
- Transactions
- Support interactions
- Complaints
- Network towers and events
- Campaigns
- Churn scores
- Monthly customer summaries

Available database schema:
{DATABASE_SCHEMA}
"""

checkpointer = InMemorySaver()

sql_agent_with_memory = create_agent(
    model=MODEL_NAME,
    tools=[execute_sql],
    system_prompt=SYSTEM_PROMPT,
    checkpointer=checkpointer,
)

print("SQL agent created successfully.")

In [ ]:
# ============================================================
# 8. Ask function and quick test
# ============================================================

def content_to_text(content):
    if isinstance(content, str):
        return content

    if isinstance(content, list):
        parts = []

        for item in content:
            if isinstance(item, dict):
                parts.append(str(item.get("text", item.get("content", item))))
            else:
                parts.append(str(item))

        return "\n".join(parts)

    return str(content)


def ask_sql_agent(question: str, thread_id: str = "zain-demo-thread-1") -> str:
    result = sql_agent_with_memory.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": question,
                }
            ]
        },
        config={
            "configurable": {
                "thread_id": thread_id
            }
        },
    )

    return content_to_text(result["messages"][-1].content)


response = ask_sql_agent("Show me the top 10 customers by revenue.")
print(response)

In [ ]:
# ============================================================
# 9. Follow-up test to confirm memory
# ============================================================

response = ask_sql_agent("Which of these customers are at high churn risk?")
print(response)

In [ ]:
# ============================================================
# 10. Optional Gradio UI for Colab
# ============================================================

import gradio as gr
from uuid import uuid4
import inspect

def create_thread_id():
    return f"zain-telecom-colab-{uuid4()}"


def normalize_history_to_messages(history):
    if history is None:
        return []

    normalized = []

    for item in history:
        if isinstance(item, dict) and "role" in item and "content" in item:
            if item["role"] in ["user", "assistant"]:
                normalized.append(
                    {
                        "role": item["role"],
                        "content": content_to_text(item["content"]),
                    }
                )

    return normalized


def chat_with_sql_agent(message, history, thread_id):
    history = normalize_history_to_messages(history)

    if not thread_id:
        thread_id = create_thread_id()

    if not message or not message.strip():
        return history, "", thread_id

    user_message = message.strip()

    try:
        result = sql_agent_with_memory.invoke(
            {
                "messages": [
                    {
                        "role": "user",
                        "content": user_message,
                    }
                ]
            },
            config={
                "configurable": {
                    "thread_id": thread_id
                }
            },
        )

        assistant_message = content_to_text(result["messages"][-1].content)

    except Exception as e:
        assistant_message = f"""
Something went wrong while running the SQL agent.

Error:

```text
{str(e)}
```
"""

    updated_history = history + [
        {
            "role": "user",
            "content": user_message,
        },
        {
            "role": "assistant",
            "content": assistant_message,
        },
    ]

    return updated_history, "", thread_id


def reset_chat():
    return [], create_thread_id()


def example_question(question):
    return question


chatbot_kwargs = {
    "value": [],
    "height": 560,
    "label": "Zain Telecom SQL Agent Chat",
    "placeholder": "Ask a telecom analytics question...",
}

# Newer Gradio versions accept type='messages'. Older versions may not.
# We always return messages format.
if "type" in inspect.signature(gr.Chatbot).parameters:
    chatbot_kwargs["type"] = "messages"


with gr.Blocks(title="Zain Telecom Customer 360 SQL Agent") as demo:
    thread_id_state = gr.State(value=create_thread_id())

    gr.Markdown(
        f"""
# Zain Telecom Customer 360 SQL Agent

Ask natural language questions about the synthetic telecom database.

**Model:** `{MODEL_NAME}`  
**Database:** `{DB_PATH}`  
**Note:** This is synthetic demo data, not real customer data.
"""
    )

    chatbot = gr.Chatbot(**chatbot_kwargs)

    with gr.Row():
        user_input = gr.Textbox(
            placeholder="Example: Which high-value customers are at high churn risk?",
            label="Your question",
            scale=8,
        )
        submit_btn = gr.Button("Ask", scale=1, variant="primary")

    clear_btn = gr.Button("New Chat / Reset Memory")

    gr.Markdown("### Example Questions")

    with gr.Row():
        ex1 = gr.Button("Top 10 customers by revenue")
        ex2 = gr.Button("High-value customers at churn risk")
        ex3 = gr.Button("Customers with unpaid invoices")
        ex4 = gr.Button("4G customers with 5G devices")

    ex1.click(example_question, inputs=[gr.State("Show me the top 10 customers by revenue.")], outputs=[user_input])
    ex2.click(example_question, inputs=[gr.State("Which high-value customers are at high churn risk?")], outputs=[user_input])
    ex3.click(example_question, inputs=[gr.State("Which customers have unpaid or overdue invoices?")], outputs=[user_input])
    ex4.click(example_question, inputs=[gr.State("Which customers are using 4G plans but have 5G-capable devices?")], outputs=[user_input])

    submit_btn.click(
        fn=chat_with_sql_agent,
        inputs=[user_input, chatbot, thread_id_state],
        outputs=[chatbot, user_input, thread_id_state],
    )

    user_input.submit(
        fn=chat_with_sql_agent,
        inputs=[user_input, chatbot, thread_id_state],
        outputs=[chatbot, user_input, thread_id_state],
    )

    clear_btn.click(
        fn=reset_chat,
        inputs=[],
        outputs=[chatbot, thread_id_state],
    )


# In Colab, share=True gives you a public temporary link.
demo.queue().launch(share=True)